In [ ]:
from pathlib import Path
from datetime import datetime

import torch
from lightning.pytorch import (
    Trainer,
    seed_everything,
)
from pytorch_lightning.loggers import WandbLogger

from patientflow.data import BrainteaserDataModule
from patientflow.models.ae import PatientEmbeddedAE, frange_cycle_linear

In [ ]:
seed_everything(42)

In [ ]:
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

ds = BrainteaserDataModule(
    42,
    32,
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year",
)
ds.setup("fit")

In [ ]:
now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
logger = WandbLogger(project="patient_flow", name=f"VAE_{now}")

In [ ]:
max_epochs = 150
trainer = Trainer(
    accelerator="gpu",
    max_epochs=max_epochs,
    logger=logger,
    enable_checkpointing=False,
    log_every_n_steps=42,
)

In [ ]:
variational_beta_weight = 1.0
variational_schedule = frange_cycle_linear(
    max_epochs * len(ds.train_dataset) // ds.batch_size,
    stop=variational_beta_weight,
)
ae = PatientEmbeddedAE(
    ds.features,
    128,  # embedding_dim
    128,  # hidden_dim
    8,  # Frequency length for encoding of continuous features
    2,  # Number of encoder heads for Transformer blocks
    64,  # Transformer Encoder Dim Feedforward 
    4,  # Number of Transformer Encoder Blocks
    64,  # Number of Channels for CNN Layers
    "attn",  # Pooling type for static feature representations
    variational=True,  # Use variational autoencoder
    variational_beta_weight=variational_beta_weight,  # Weight for the variational loss
    variational_beta_schedule="decreasing",  # Schedule for the variational loss
)

In [ ]:
trainer.fit(ae, datamodule=ds)

In [ ]:
logger.experiment.finish()

In [ ]:
torch.save(ae, f"vae_{now}.pt")